## 1. Setup

In [1]:
import json
import sys
from pathlib import Path
import joblib
from datetime import date

import pandas as pd

from sklearn.base import clone
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error


PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


from src.data.columns import PATIENT_COL, DIAGNOSIS_COL
from src.data.ml_dataset_specs import UCVA_TARGET

from src.modeling.model_inputs import prepare_train_test_model_inputs
from src.modeling.configs import ElasticNetConfig
from src.modeling.pipelines import (
    make_dummy_pipeline,
    make_elasticnet_pipeline,
    make_linear_pipeline,
    make_log_target_pipeline,
)


DATA_PATH = PROJECT_ROOT / "data" / "ml" / "ucva_exam1_base_2024-11-02.xlsx"
CANDIDATES_PATH = PROJECT_ROOT / "artifacts" / "model_selection" / "final_candidates.json"
FINAL_MODEL_DIR = PROJECT_ROOT / "artifacts" / "final_models"

TEST_SIZE = 0.2
RANDOM_STATE = 42

## 2. Train/Test Data Preparation

In [2]:
df = pd.read_excel(DATA_PATH)
df.shape

(399, 80)

In [3]:
inputs = prepare_train_test_model_inputs(
    df=df,
    target_col=UCVA_TARGET,
    group_col=PATIENT_COL,
    exclude_feature_cols=[DIAGNOSIS_COL],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

inputs.train.X.shape, inputs.test.X.shape

((319, 77), (80, 77))

## 3. Candidates Evaluation

In [4]:
def build_estimator_from_config(config: dict):
    factory = config["pipeline_factory"]
    params = config.get("params", {})

    if factory == "make_dummy_pipeline":
        return make_dummy_pipeline(**params)

    if factory == "make_elasticnet_pipeline":
        en_config = ElasticNetConfig(**params)
        return make_elasticnet_pipeline(en_config)

    if factory == "make_log_target_pipeline(make_linear_pipeline)":
        return make_log_target_pipeline(make_linear_pipeline())

    raise ValueError(f"Unknown pipeline_factory: {factory}")

In [5]:
with open(CANDIDATES_PATH, "r", encoding="utf-8") as f:
    candidates_config = json.load(f)
    
final_results = []

for candidate_name, config in candidates_config.items():
    estimator = build_estimator_from_config(config)

    features = config["features"]

    if features == "all":
        selected_features = list(inputs.train.X.columns)
    else:
        selected_features = features

    X_train = inputs.train.X[selected_features]
    y_train = inputs.train.y

    X_test = inputs.test.X[selected_features]
    y_test = inputs.test.y

    model = clone(estimator)
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    mae_train = mean_absolute_error(y_train, y_pred_train)
    mae_test = mean_absolute_error(y_test, y_pred_test)

    rmse_train = root_mean_squared_error(y_train, y_pred_train)
    rmse_test = root_mean_squared_error(y_test, y_pred_test)

    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    final_results.append({
        "candidate": candidate_name,
        "n_features": len(selected_features),
        
        "mae_train": mae_train,
        "mae_test": mae_test,
        "mae_gap": mae_test - mae_train,

        "rmse_train": rmse_train,
        "rmse_test": rmse_test,
        "rmse_gap": rmse_test - rmse_train,
        
        "r2_train": r2_train,
        "r2_test": r2_test,
        "r2_gap": r2_train - r2_test,
    })

final_results_df = (
    pd.DataFrame(final_results)
    .sort_values(["mae_test"], ascending=[True])
    .reset_index(drop=True)
)

final_results_df

,candidate,n_features,mae_train,mae_test,mae_gap,rmse_train,rmse_test,rmse_gap,r2_train,r2_test,r2_gap
0,Stepwise_LR_log,9,0.064518,0.101283,0.036764,0.164419,0.233511,0.069092,0.379964,0.298590,0.081374
1,Dummy_median,77,0.085956,0.129000,0.043044,0.224219,0.305659,0.081441,-0.153066,-0.201795,0.048730
2,ENSelector_EN_alpha_0.35_l1_0.1,6,0.095211,0.141217,0.046006,0.182588,0.252044,0.069456,0.235367,0.182839,0.052528


## 4. Final Model Selection and Saving

In [6]:
final_model_name = "Stepwise_LR_log"
final_config = candidates_config[final_model_name]

In [7]:
final_config["features"]

['Рефрактометрія циклоплегія Sphera',
 'Рефрактометрія циклоплегія Cylynder',
 'Axial length (mm)',
 'Вік',
 'Кератометрія циклоплегії Cyl',
 'Рефрактометрія без циклоплегії Sphera',
 'Pupil diameter (mm)',
 'Стать: 1 - ч, 2 - ж',
 'Flat meridian K1 (D)']

In [8]:
FEATURE_RENAME_MAP = {
    "Рефрактометрія циклоплегія Sphera": "cyclo_sphere",
    "Axial length (mm)": "axial_length",
    "Вік": "age",
    "Рефрактометрія циклоплегія Cylynder": "cyclo_cylinder",
    "Рефрактометрія без циклоплегії Sphera": "manifest_sphere",
    "Стать: 1 - ч, 2 - ж": "sex",
    "Кератометрія циклоплегії Cyl": "keratometry_cylinder",
    "Pupil diameter (mm)": "pupil_diameter",
    "Flat meridian K1 (D)": "k1_flat",
}

In [9]:
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_PATH = FINAL_MODEL_DIR / "ucva_stepwise_linear_log_v1.joblib"


final_estimator = build_estimator_from_config(final_config)

final_features = final_config["features"]
if final_features == "all":
    final_features = list(inputs.train.X.columns)

missing_features = set(final_features) - set(FEATURE_RENAME_MAP)

if missing_features:
    raise ValueError(
        "Some final features are missing in FEATURE_RENAME_MAP: "
        f"{sorted(missing_features)}"
    )

X_final = df[final_features].copy()
X_final = X_final.rename(columns=FEATURE_RENAME_MAP)
y_final = df[UCVA_TARGET].copy()

final_estimator.fit(X_final, y_final)

artifact = {
    "pipeline": final_estimator,
    "feature_names": list(X_final.columns),
    "feature_rename_map": FEATURE_RENAME_MAP,
    "target_name": UCVA_TARGET,
    "model_name": final_model_name,
    "model_version": "v1",
    "training_date": date.today().isoformat(),
    "training_ranges": {
        feature: {
            "min": float(X_final[feature].min()),
            "max": float(X_final[feature].max()),
        }
        for feature in X_final.columns
    },
}

joblib.dump(artifact, FINAL_MODEL_PATH)


['d:\\professional\\code\\python\\visual-acuity-ml\\artifacts\\final_models\\ucva_stepwise_linear_log_v1.joblib']